# 04 — Transfer Analysis

Interactive analysis of Phase 3 results: shared mechanism testing.

Goals:
- Visualize cross-condition transfer rates as a function of number of features clamped
- Plot the cosine similarity and canonical angles between failure mode directions across layers
- Identify the layer(s) of maximum geometric alignment
- Assess overall evidence for/against the shared override circuit hypothesis

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PHASE3_RUN = 'FILL_IN_RUN_ID'
PHASE3_DIR = Path('../experiments/results/phase3')

In [ ]:
# Load transfer results
with open(PHASE3_DIR / PHASE3_RUN / 'transfer_cot_to_syco.json') as f:
    cot_to_syco = json.load(f)
with open(PHASE3_DIR / PHASE3_RUN / 'transfer_syco_to_cot.json') as f:
    syco_to_cot = json.load(f)
with open(PHASE3_DIR / PHASE3_RUN / 'run_summary.json') as f:
    summary = json.load(f)

print(f"Shared mechanism evidence: {summary['shared_mechanism_evidence'].upper()}")
print(f"Interpretation: {summary['interpretation']}")

In [ ]:
# Transfer rate as a function of features clamped
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

c2s_n = [r['n_features_clamped'] for r in cot_to_syco]
c2s_rate = [r['transfer_rate'] for r in cot_to_syco]
c2s_syco_delta = [r['sycophancy_delta'] for r in cot_to_syco]

s2c_n = [r['n_features_clamped'] for r in syco_to_cot]
s2c_rate = [r['transfer_rate'] for r in syco_to_cot]
s2c_faith_delta = [r['faithfulness_delta'] for r in syco_to_cot]

ax1.plot(c2s_n, c2s_rate, 'o-', color='#d6604d', label='transfer rate')
ax1.plot(c2s_n, c2s_syco_delta, 's--', color='#f4a582', label='sycophancy reduction')
ax1.axhline(0.3, color='grey', linestyle=':', linewidth=1, label='moderate threshold')
ax1.set_xlabel('Features clamped (from CoT condition)')
ax1.set_ylabel('Effect on sycophancy prompts')
ax1.set_title('CoT features → Sycophancy prompts')
ax1.legend(fontsize=9)

ax2.plot(s2c_n, s2c_rate, 'o-', color='#2166ac', label='transfer rate')
ax2.plot(s2c_n, s2c_faith_delta, 's--', color='#92c5de', label='faithfulness improvement')
ax2.axhline(0.3, color='grey', linestyle=':', linewidth=1, label='moderate threshold')
ax2.set_xlabel('Features clamped (from sycophancy condition)')
ax2.set_title('Sycophancy features → CoT prompts')
ax2.legend(fontsize=9)

plt.suptitle('Cross-condition transfer: evidence for shared override mechanism', fontsize=11)
plt.tight_layout()
plt.savefig('../paper/figures/transfer_rates.png', dpi=150)
plt.show()

In [ ]:
# Representational geometry: cosine similarity and canonical angles across layers
with open(PHASE3_DIR / PHASE3_RUN / 'geometry.json') as f:
    geometry = json.load(f)

layers = [g['layer'] for g in geometry]
cosine_sims = [g['cosine_similarity'] for g in geometry]
mean_angles = [g['mean_canonical_angle_deg'] for g in geometry]
min_angles = [g['min_canonical_angle_deg'] for g in geometry]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(layers, cosine_sims, 'o-', color='#2166ac', linewidth=1.5, markersize=4)
ax1.axhline(0, color='grey', linestyle='--', linewidth=1)
peak_layer = summary.get('peak_alignment_layer')
if peak_layer is not None:
    ax1.axvline(peak_layer, color='red', linestyle=':', label=f'Peak alignment: L{peak_layer}')
    ax1.legend()
ax1.set_ylabel('Cosine similarity')
ax1.set_title('Cosine similarity between unfaithfulness and sycophancy directions')
ax1.set_ylim(-1, 1)

ax2.fill_between(layers, min_angles, mean_angles, alpha=0.3, color='#d6604d', label='[min, mean] angle range')
ax2.plot(layers, mean_angles, 'o-', color='#d6604d', linewidth=1.5, markersize=4)
ax2.axhline(90, color='grey', linestyle='--', linewidth=1, label='Orthogonal (90°)')
ax2.set_xlabel('Model layer')
ax2.set_ylabel('Canonical angle (degrees)')
ax2.set_title('Canonical angles between failure mode subspaces (lower = more aligned)')
ax2.legend()
ax2.set_ylim(0, 95)

plt.tight_layout()
plt.savefig('../paper/figures/geometry_across_layers.png', dpi=150)
plt.show()

In [ ]:
# Summary table
import pandas as pd

df_cot = pd.DataFrame(cot_to_syco)[['n_features_clamped', 'transfer_rate', 'sycophancy_delta', 'accuracy_delta']]
df_cot.columns = ['n_features', 'transfer_rate', 'sycophancy_delta', 'accuracy_delta']
df_cot['direction'] = 'CoT → Sycophancy'

df_syco = pd.DataFrame(syco_to_cot)[['n_features_clamped', 'transfer_rate', 'faithfulness_delta', 'accuracy_delta']]
df_syco = df_syco.rename(columns={'faithfulness_delta': 'sycophancy_delta'})
df_syco['direction'] = 'Sycophancy → CoT'

combined = pd.concat([df_cot, df_syco], ignore_index=True)
print(combined.to_string(index=False))